**Indexing**
1. Load the pdf data using PyMyPDFLoader from langchain_community.document_loaders
2. Split the text/docs into chunks using Recursive... from langchain_textsplitters
3. Generate text embeddings with:
    - OpenAI text-embedding-3-small
    - Testing with ollama:
      Embedding model: ollama pull hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
      Language model: ollama pull hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF
4. Store embeddings in a vector database:
    - Pinecone
    - FAISS
    - Chromadb

**Retrieval**
1. Translate users question to embeddings
2. Retrieve relevant chunks according to the question as context
3. Add a system prompt (a prompt for the llm's behavior), the context and the user's question (again)
4. Push this result to the llm to generate a grounded answer


### Load PDF

In [ ]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

loader = PyMuPDF4LLMLoader(file_path="./data/admission_requirement.pdf")
docs = loader.load()
print(docs)

### Split PDF into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc_splitter = RecursiveCharacterTextSplitter(
  chunk_size=500,
  chunk_overlap=100
)
chunked_docs = doc_splitter.split_documents(docs)

print(f"Number of chunks: {len(chunked_docs)}")

for i, chunk in enumerate(chunked_docs, start=1):
  print(f"\nChunk {i}:")
  print(f"metadata: {chunk.metadata}\n")
  print(f"content: '{chunk.page_content[:150]}'")


### Translate Chunks into embeddings

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
  model="hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:latest"
)

In [ ]:
from langchain_chroma import Chroma

chroma_vector_db = Chroma.from_documents(
  documents=chunked_docs,
  embedding=embeddings,
  persist_directory="./chroma_vector_db"
)
print(f"Chroma Vector Database with {chroma_vector_db._collection.count()} embeddings")


### Retrieval

In [ ]:
# Store into vectordb
retriever = chroma_vector_db.as_retriever(
  search_kwargs={"k":2}
)

retrieved_docs = retriever.invoke("What is the cutoff point Sociology")
print(retrieved_docs)

### Generation

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
  model="hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF"
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are KNUST Admissions Assistant, an AI agent that helps prospective students find 
cutoff points and admission requirements for programs at Kwame Nkrumah University of 
Science and Technology (KNUST).

## Your Role
- Help users identify the correct program based on the subjects/field they want to study.
- Provide the cutoff point(s) and entry requirements for that program, strictly using 
  the retrieved context provided to you.
- Clarify ambiguous program names before answering (e.g. "BSc. Computer Science" vs 
  "BSc. Computer Engineering" vs "Information Technology").

## Grounding Rules
- Only answer using information present in the retrieved context. Never rely on prior 
  knowledge of KNUST cutoffs, as these change every admission cycle.
- If the retrieved context does not contain the answer, say so plainly and suggest the 
  user check the official KNUST admissions portal or contact the admissions office — 
  do not guess or estimate a cutoff point.
- If multiple years of cutoff data appear in the context, default to the most recent 
  year unless the user asks otherwise, and state which year you're quoting.
- Always distinguish between aggregate cutoff points for different qualification tracks 
  if the context includes more than one (e.g. WASSCE vs SSSCE, or Arts vs Science 
  aggregate systems), and ask the user which applies to them if unclear.

## Handling Ambiguity
- If a user names a subject area rather than a specific program (e.g. "I want to study 
  medicine-related courses"), list the matching programs from the retrieved context and 
  ask which one they mean before giving requirements.
- If a program name is misspelled or informally phrased, match it to the closest known 
  program in the context and confirm with the user.

## Response Format
- State the program name clearly.
- Give the cutoff point (and the year it applies to).
- List core entry requirements (e.g. required WASSCE core/elective subjects, grade 
  thresholds).
- Note any additional requirements (interviews, portfolios, quotas) if present in context.
- Keep responses concise and structured — use short lists over long paragraphs.

## Tone
- Warm, encouraging, and clear — many users are first-time applicants or parents. 
  Avoid jargon; explain grading systems (e.g. aggregate scoring) briefly if relevant.

## Boundaries
- Do not provide admission advice outside KNUST (e.g. other universities) unless asked 
  for a comparison, and even then, only using retrieved context.
- Do not fabricate cutoff numbers, subject combinations, or deadlines under any 
  circumstances. When context is insufficient, be transparent about the gap."""

prompt = ChatPromptTemplate.from_messages([
      ("system", system_prompt + "\n\nRetrived context:\n{context}"),
      ("human", "{question}"),
  ])

# prompt = ChatPromptTemplate.from_template(system_prompt)
# print(prompt)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm |StrOutputParser()
)

response = chain.invoke("What's the requirement for reading Mathematics")
print(response)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI


SYSTEM_PROMPT = """You are KNUST Admissions Assistant, an AI agent that helps prospective students find 
cutoff points and admission requirements for programs at Kwame Nkrumah University of 
Science and Technology (KNUST).

## Your Role
- Help users identify the correct program based on the subjects/field they want to study.
- Provide the cutoff point(s) and entry requirements for that program, strictly using 
  the retrieved context provided to you.
- Clarify ambiguous program names before answering (e.g. "BSc. Computer Science" vs 
  "BSc. Computer Engineering" vs "Information Technology").

## Grounding Rules
- Only answer using information present in the retrieved context. Never rely on prior 
  knowledge of KNUST cutoffs, as these change every admission cycle.
- If the retrieved context does not contain the answer, say so plainly and suggest the 
  user check the official KNUST admissions portal or contact the admissions office — 
  do not guess or estimate a cutoff point.
- If multiple years of cutoff data appear in the context, default to the most recent 
  year unless the user asks otherwise, and state which year you're quoting.
- Always distinguish between aggregate cutoff points for different qualification tracks 
  if the context includes more than one (e.g. WASSCE vs SSSCE, or Arts vs Science 
  aggregate systems), and ask the user which applies to them if unclear.

## Handling Ambiguity
- If a user names a subject area rather than a specific program (e.g. "I want to study 
  medicine-related courses"), list the matching programs from the retrieved context and 
  ask which one they mean before giving requirements.
- If a program name is misspelled or informally phrased, match it to the closest known 
  program in the context and confirm with the user.

## Response Format
- State the program name clearly.
- Give the cutoff point (and the year it applies to).
- List core entry requirements (e.g. required WASSCE core/elective subjects, grade 
  thresholds).
- Note any additional requirements (interviews, portfolios, quotas) if present in context.
- Keep responses concise and structured — use short lists over long paragraphs.

## Tone
- Warm, encouraging, and clear — many users are first-time applicants or parents. 
  Avoid jargon; explain grading systems (e.g. aggregate scoring) briefly if relevant.

## Boundaries
- Do not provide admission advice outside KNUST (e.g. other universities) unless asked 
  for a comparison, and even then, only using retrieved context.
- Do not fabricate cutoff numbers, subject combinations, or deadlines under any 
  circumstances. When context is insufficient, be transparent about the gap."""


# ---------------------------------------------------------------------------
# 1. The prompt template — same as before, with two placeholders LangChain
#    will fill in automatically: {context} and {question}
# ---------------------------------------------------------------------------
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT + "\n\nRetrieved context:\n{context}"),
    ("human", "{question}"),
])

# ---------------------------------------------------------------------------
# 2. The LLM — LangChain's wrapper around the OpenAI chat completions API
# ---------------------------------------------------------------------------
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

# ---------------------------------------------------------------------------
# 3. A helper that formats retrieved chunks into a numbered context string,
#    exactly like the f-string join did in the plain Python version.
# ---------------------------------------------------------------------------
def format_docs(chunks) -> str:
    return "\n\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(chunks))


def retrieve_and_format(question: str) -> str:
    chunks = retriever.retrieve(question)
    return format_docs(chunks)


# ---------------------------------------------------------------------------
# 4. The chain (LCEL — "LangChain Expression Language").
#    Read the pipe `|` as "pass the output of the left side into the right
#    side." This is the part that replaces manual string-building.
# ---------------------------------------------------------------------------
rag_chain = (
    {
        # "context" key: run retrieve_and_format on the incoming question
        "context": RunnableLambda(retrieve_and_format),
        # "question" key: pass the incoming question through unchanged
        "question": RunnablePassthrough(),
    }
    | prompt              # fills {context} and {question} into the template
    | llm                 # sends the filled prompt to the chat model
    | StrOutputParser()   # extracts the plain string from the model response
)


def answer_question(query: str) -> str:
    """Full RAG via LangChain: retrieval + grounded generation."""
    return rag_chain.invoke(query)


if __name__ == "__main__":
    test_questions = [
        "what's the cutoff for computer science?",
        "I want to study something medicine related, what are my options?",
        "can I do architecture with visual art background?",
    ]
    for q in test_questions:
        print(f"Q: {q}")
        print(f"A: {answer_question(q)}\n")